# Tiled attention and online softmax

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/zhubarb/sesen_ai_ml_tutorials/blob/main/notebooks/deep-learning/flash_attention.ipynb)

This notebook runs on CPU. It generates all data and performs the experiment; no model download or API credentials are needed. GPU timing is optional for attention.

Dependencies: Python 3.10+, torch, numpy, scipy, matplotlib. In Colab these are normally installed. If needed run `%pip install torch numpy scipy matplotlib` before the imports.

Tested on Python 3.12 with PyTorch 2.14.0+rocm10.1.0, NumPy 2.5.3, SciPy 1.18.1 and Matplotlib 3.11.2. The ROCm suffix identifies the development machine build; retain the PyTorch build supplied by Colab for its CPU or NVIDIA runtime. Different library versions can change timings and floating-point details.

In [ ]:
import math, time
import numpy as np
import torch
from torch import nn
from torch.nn import functional as F
from scipy.optimize import linear_sum_assignment
import matplotlib.pyplot as plt
torch.set_num_threads(4)

## Implementation

The functions below are the complete implementation. Fixed seeds make the inputs repeatable; timings depend on the machine. Read the accompanying tutorial for derivations and limits.

In [ ]:
def dense_attention(q,k,v):
    return (q@k.transpose(-2,-1)/math.sqrt(q.shape[-1])).softmax(-1)@v

def tiled_attention(q,k,v,block=64):
    """Unmasked exact attention; ordinary tensor ops, no custom fused kernel."""
    outputs=[]
    for qi in q.split(block,dim=-2):
        shape=(*qi.shape[:-1],1)
        m=torch.full(shape,-torch.inf,dtype=q.dtype,device=q.device)
        ell=torch.zeros(shape,dtype=q.dtype,device=q.device)
        acc=torch.zeros_like(qi)
        for kj,vj in zip(k.split(block,dim=-2),v.split(block,dim=-2)):
            scores=qi@kj.transpose(-2,-1)/math.sqrt(q.shape[-1])
            new_m=torch.maximum(m,scores.max(-1,keepdim=True).values)
            alpha=(m-new_m).exp()
            p=(scores-new_m).exp()
            acc=alpha*acc+p@vj
            ell=alpha*ell+p.sum(-1,keepdim=True)
            m=new_m
        outputs.append(acc/ell)
    return torch.cat(outputs,dim=-2)

def flash_experiment(device='cpu'):
    start=time.perf_counter();torch.manual_seed(117)
    q,k,v=[torch.randn(1,2,19,8,dtype=torch.float64,requires_grad=True) for _ in range(3)]
    a=dense_attention(q,k,v);b=tiled_attention(q,k,v,7)
    probe=torch.randn_like(a)
    ga=torch.autograd.grad((a*probe).sum(),(q,k,v),retain_graph=True)
    gb=torch.autograd.grad((b*probe).sum(),(q,k,v))
    errors={'output':float((a-b).detach().abs().max()),'gradient':max(float((x-y).abs().max()) for x,y in zip(ga,gb))}
    assert errors['output']<1e-12 and errors['gradient']<1e-12
    records=[];device_errors=[]
    for dev in dict.fromkeys(['cpu',device]):
      for n in [128,256,512,1024,2048]:
        inputs=[torch.randn(1,2,n,32,device=dev,dtype=torch.float32) for _ in range(3)]
        with torch.no_grad():
          expected=dense_attention(*inputs)
          for label,check in [('tiled',tiled_attention),('sdpa_auto',F.scaled_dot_product_attention)]:
            actual=check(*inputs);err=float((actual-expected).abs().max());assert torch.allclose(actual,expected,atol=2e-5,rtol=2e-4)
            device_errors.append(dict(device=dev,n=n,method=label,max_abs=err))
          del expected,actual
        for name,fn in [('dense',dense_attention),('tiled',tiled_attention),('sdpa_auto',F.scaled_dot_product_attention)]:
          with torch.no_grad():
            for _ in range(2):fn(*inputs)
            if dev=='cuda':torch.cuda.synchronize();torch.cuda.reset_peak_memory_stats();base=torch.cuda.memory_allocated()
            ts=[]
            for _ in range(5):
              t=time.perf_counter();result=fn(*inputs)
              if dev=='cuda':torch.cuda.synchronize()
              ts.append((time.perf_counter()-t)*1000)
              del result
            peak=(torch.cuda.max_memory_allocated()-base) if dev=='cuda' else None
          records.append(dict(device=dev,n=n,method=name,median_ms=float(np.median(ts)),peak_extra_bytes=peak))
    backends={}
    if device=='cuda':
      from torch.nn.attention import sdpa_kernel,SDPBackend
      for name,be in [('FLASH_ATTENTION',SDPBackend.FLASH_ATTENTION),('EFFICIENT_ATTENTION',SDPBackend.EFFICIENT_ATTENTION),('MATH',SDPBackend.MATH)]:
        try:
          with sdpa_kernel(be):F.scaled_dot_product_attention(*inputs);torch.cuda.synchronize()
          backends[name]='supported for measured float32 shape'
        except Exception as e:backends[name]=str(e).splitlines()[0]
    # Per-tile online softmax accumulator, one query, exactly the benchmark recurrence.
    rng=np.random.default_rng(117);scores=rng.normal(0,5,96);values=rng.normal(size=(96,2));m=-np.inf;ell=0.;acc=np.zeros(2);trace=[]
    for j in range(0,96,8):
      z=scores[j:j+8];new=max(m,z.max());alpha=np.exp(m-new);p=np.exp(z-new)
      acc=alpha*acc+p@values[j:j+8];ell=alpha*ell+p.sum();m=new
      trace.append({'keys':j+8,'max':float(m),'sum':float(ell),'output':(acc/ell).tolist()})
    return dict(errors=errors,device_errors=device_errors,records=records,backends=backends,trace=trace,seconds=time.perf_counter()-start,torch=torch.__version__)

## Experiment and checks

Assertions check correctness before the comparison. The sparse-autoencoder experiment uses held-out activations, and DPO evaluates fresh preference labels on known contexts. Training settings are deliberately small.

In [ ]:
result = flash_experiment('cpu')
print('Elapsed seconds:', result['seconds'])
print(result.get('errors', result['records'][0]))

## Measured figures

Each point comes from the experiment above. For attention on CPU the memory panel is labelled analytical score storage, because CPU allocator peaks are not measured. GPU results use allocator statistics.

In [ ]:
def flash_plots(result):
    fig,axes=plt.subplots(1,2,figsize=(11,4))
    for name in ['dense','tiled','sdpa_auto']:
      rs=[r for r in result['records'] if r['device']=='cuda' and r['method']==name]
      if not rs:rs=[r for r in result['records'] if r['device']=='cpu' and r['method']==name]
      axes[0].plot([r['n'] for r in rs],[r['median_ms'] for r in rs],'o-',label=name)
      if rs[0]['peak_extra_bytes'] is not None:axes[1].plot([r['n'] for r in rs],[r['peak_extra_bytes']/2**20 for r in rs],'o-',label=name)
    axes[0].set(xlabel='Sequence length',ylabel='Median forward latency (ms)',yscale='log');axes[0].legend()
    if any(r['peak_extra_bytes'] is not None for r in result['records']):axes[1].set(xlabel='Sequence length',ylabel='Extra allocated GPU memory (MiB)',yscale='log');axes[1].legend()
    else:
      n=np.array([128,256,512,1024,2048]);axes[1].plot(n,2*n*n*4/2**20,label='Dense score tensor only');axes[1].plot(n,2*64*64*4/2**20*np.ones_like(n),label='One tiled score block');axes[1].set(xlabel='Sequence length',ylabel='Analytical score storage (MiB)');axes[1].legend()
    fig.tight_layout();return fig

flash_plots(result)
plt.show()

## Exercises

1. Change block size to 7 and verify non-divisible sequence lengths.
2. Add a causal mask and handle all-masked blocks without NaNs.
3. If a GPU is available, run `flash_experiment("cuda")` and inspect forced-backend results.
4. Compare float32 and float64 errors without asserting bitwise equality.

## Reference and scope

Primary reference: https://arxiv.org/abs/2205.14135

This is a controlled teaching experiment. The attention implementation uses ordinary PyTorch operations; the autoencoder uses planted synthetic features; DPO uses categorical responses rather than a language model.